In [1]:
from pyspark.sql.functions import *

StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 3, Finished, Available, Finished, False)

In [2]:

# ================================
# ADDED: PERFORMANCE SETUP
# ================================
import time, json

metrics = {}
start_time = time.time()

def log_metric(name, value):
    metrics[name] = value

def count_df(df):
    return df.count()

def estimate_size_mb(df):
    return df.rdd.map(lambda r: len(str(r))).sum() / (1024*1024)


StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 4, Finished, Available, Finished, False)

In [3]:
df_customers = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Customers/ShoppingMart_customers.csv")
df_orders = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Orders/ShoppingMart_orders.csv")
df_products = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Products/ShoppingMart_products.csv")
df_reviews = spark.read.json("Files/ShoppingMart_Bronze_Reviews/ShoppingMart_review.json")
df_social = spark.read.json("Files/ShoppingMart_Bronze_Social_Media/ShoppingMart_social_media.json")
df_weblogs = spark.read.json("Files/ShoppingMart_Bronze_Web_Logs/ShoppingMart_web_logs.json")


StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 5, Finished, Available, Finished, False)

In [4]:
# ================================
# ADDED: INPUT METRICS
# ================================
log_metric("customers_input_rows", count_df(df_customers))
log_metric("orders_input_rows_raw", count_df(df_orders))
log_metric("products_input_rows", count_df(df_products))
log_metric("reviews_input_rows", count_df(df_reviews))
log_metric("social_input_rows", count_df(df_social))
log_metric("weblogs_input_rows", count_df(df_weblogs))

orders_before_clean = count_df(df_orders)

StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 6, Finished, Available, Finished, False)

In [5]:
# DATA CLEANING AND ENRICHING	
df_orders = df_orders.dropna(subset = ["OrderID", "CustomerID", "ProductID", "OrderDate", "TotalAmount"])
df_orders = df_orders.withColumn("OrderDate", to_date(col("OrderDate")))
# display(df_orders)

StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 7, Finished, Available, Finished, False)

In [6]:
# ================================
# ADDED: CLEANING METRICS
# ================================
orders_after_clean = count_df(df_orders)
log_metric("orders_after_clean", orders_after_clean)
log_metric("orders_dropped_rows", orders_before_clean - orders_after_clean)

join_input_rows = count_df(df_orders)

StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 8, Finished, Available, Finished, False)

In [7]:

# JOIN WITH PRODUCTS & CUSTOMERS
df_orders = df_orders \
    .join (df_customers, on = 'CustomerID', how = "inner") \
    .join (df_products, on = 'ProductID', how = "inner")


StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 9, Finished, Available, Finished, False)

In [8]:
# ================================
# ADDED: JOIN METRICS
# ================================
df_orders.cache()
join_output_rows = count_df(df_orders)

log_metric("join_input_rows", join_input_rows)
log_metric("join_output_rows", join_output_rows)

StatementMeta(, 51310305-7582-41fe-a33f-e11bf42900dc, 10, Finished, Available, Finished, False)

In [55]:

# WRITE DATA TO SILVER LAYER	
df_orders.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Orders/ShoppingMart_customers_orderdata")


StatementMeta(, d60bfd65-7f34-4185-a5ba-c82f7bd1f759, 35, Finished, Available, Finished, False)

In [56]:

df_reviews.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Reviews/ShoppingMart_review")
df_social.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Social_Media/ShoppingMart_social_media")
df_weblogs.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Web_Logs/ShoppingMart_web_logs")

StatementMeta(, d60bfd65-7f34-4185-a5ba-c82f7bd1f759, 36, Finished, Available, Finished, False)

In [57]:
# ================================
# ADDED: OUTPUT + PERFORMANCE
# ================================
silver_output_rows = count_df(df_orders)
silver_output_size_mb = estimate_size_mb(df_orders)

log_metric("silver_output_rows", silver_output_rows)
log_metric("silver_output_size_mb", silver_output_size_mb)

end_time = time.time()

execution_time = end_time - start_time
throughput = silver_output_rows / execution_time if execution_time > 0 else 0

log_metric("execution_time_sec", execution_time)
log_metric("throughput_rows_per_sec", throughput)

spark.createDataFrame([(json.dumps(metrics),)], ["metrics"]) \
    .write.mode("append") \
    .text("Files/ETL_Metrics/Silver_metrics")

print("✅ Silver metrics captured")

StatementMeta(, d60bfd65-7f34-4185-a5ba-c82f7bd1f759, 37, Finished, Available, Finished, False)

✅ Silver metrics captured


In [58]:
# # ================================
# # PRINT SILVER TABLE SCHEMAS
# # ================================

# print("===== SILVER TABLE SCHEMAS =====\n")

# silver_paths = {
#     "df_orders (Silver Orders)": "Files/ShoppingMart_Silver_Orders/ShoppingMart_customers_orderdata",
#     "df_reviews (Silver Reviews)": "Files/ShoppingMart_Silver_Reviews/ShoppingMart_review",
#     "df_social (Silver Social)": "Files/ShoppingMart_Silver_Social_Media/ShoppingMart_social_media",
#     "df_weblogs (Silver Weblogs)": "Files/ShoppingMart_Silver_Web_Logs/ShoppingMart_web_logs"
# }

# for name, path in silver_paths.items():
#     print(f"\n--- {name} ---")
#     df = spark.read.parquet(path)
#     df.printSchema()

StatementMeta(, d60bfd65-7f34-4185-a5ba-c82f7bd1f759, 38, Finished, Available, Finished, False)

In [ ]:
# from pyspark.sql.functions import *

# # ================================
# # READ
# # ================================
# path_orders  = "Files/ShoppingMart_Silver_Orders/ShoppingMart_customers_orderdata"
# path_reviews = "Files/ShoppingMart_Silver_Reviews/ShoppingMart_review"
# path_social  = "Files/ShoppingMart_Silver_Social_Media/ShoppingMart_social_media"
# path_weblogs = "Files/ShoppingMart_Silver_Web_Logs/ShoppingMart_web_logs"

# df_orders  = spark.read.parquet(path_orders)
# df_reviews = spark.read.parquet(path_reviews)
# df_social  = spark.read.parquet(path_social)
# df_weblogs = spark.read.parquet(path_weblogs)

# # ================================
# # BEFORE COUNT
# # ================================
# print("BEFORE COUNTS")
# print("Orders  :", df_orders.count())
# print("Reviews :", df_reviews.count())
# print("Social  :", df_social.count())
# print("Weblogs :", df_weblogs.count())

# # ================================
# # REMOVE 10% (BREAK LINEAGE)
# # ================================
# df_orders_new  = df_orders.sample(0.9, seed=42).cache()
# df_reviews_new = df_reviews.sample(0.9, seed=42).cache()
# df_social_new  = df_social.sample(0.9, seed=42).cache()
# df_weblogs_new = df_weblogs.sample(0.9, seed=42).cache()

# # force materialization
# df_orders_new.count()
# df_reviews_new.count()
# df_social_new.count()
# df_weblogs_new.count()

# # ================================
# # AFTER COUNT
# # ================================
# print("\nAFTER COUNTS")
# print("Orders  :", df_orders_new.count())
# print("Reviews :", df_reviews_new.count())
# print("Social  :", df_social_new.count())
# print("Weblogs :", df_weblogs_new.count())

# # ================================
# # WRITE (SAFE OVERWRITE)
# # ================================
# spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# df_orders_new.write.mode("overwrite").parquet(path_orders)
# df_reviews_new.write.mode("overwrite").parquet(path_reviews)
# df_social_new.write.mode("overwrite").parquet(path_social)
# df_weblogs_new.write.mode("overwrite").parquet(path_weblogs)

# print("\n✅ 10% deleted successfully")

StatementMeta(, d60bfd65-7f34-4185-a5ba-c82f7bd1f759, 39, Finished, Available, Finished, False)

BEFORE COUNTS
Orders  : 2939
Reviews : 3000
Social  : 3000


Weblogs : 3000



AFTER COUNTS
Orders  : 2664
Reviews : 2715
Social  : 2715
Weblogs : 2715



✅ 10% deleted successfully
